# Intra-US MRIOT, v2 construction (margins aggregated with goods flows)

Clean notebook using the **aggregation method**: trade/transport margins are folded into the
goods flows instead of being routed separately.

- inter-state: a single combined bilateral flow `T_new` from `reconstruct_bilateral_2`, whose
  column target is `nd0 + nm0` (import demand + national margin demand together);
- local: `xd0 = dd0 + dm0` as the local source (domestic margins folded in);
- a single use-share set (`compute_use_shares_2`) -- goods and margins share the same
  allocation key.

Contrast with `v3_construction`, which separates the trade and margin flows and routes margins
through the `md0` chain (good -> margin type -> carrier good -> buyer).

Structure: **Setup -> Functions -> Construction -> Diagnostic -> Analyse**.

## Setup

# Librairies

#### Installations

In [ ]:
from paths import ROOT
import sys
!{sys.executable} -m pip install gdx2py

In [ ]:
import sys
!{sys.executable} -m pip install gamspy-base

#### Imports

In [ ]:
from gdx2py import GdxFile
import gamspy_base
import os
import pandas as pd
from gdx2py.gams import GAMSParameter, GAMSSet
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import product

In [ ]:
gamspybase_directory = gamspy_base.directory
print(gamspybase_directory)

In [ ]:
path_windc_gdx = str(ROOT / "data/raw/GTAPWiNDC/data/core/WiNDCdatabase.gdx")

In [ ]:
gdx = GdxFile(path_windc_gdx, gams_dir=gamspybase_directory)
print(list(gdx))

In [ ]:
# ── 1. Load all parameters ───────────────────────────────────────────────────
params = {}
for name, obj in gdx:
    if isinstance(obj, GAMSParameter):
        s = obj.to_pandas()
        if s is not None and len(s) > 0:
            df = s.reset_index()
            df.columns = list(df.columns[:-1]) + ['value']
            params[name] = df
# params contains all years and regions. We will filter it later when we need to build the IOT for a specific year and region.

In [ ]:
# Regions: union of all states present in xn0_ or nd0_
all_xn0 = set(params['xn0_']['r'].unique())
all_nd0 = set(params['nd0_']['r'].unique())
regions = sorted(all_xn0 | all_nd0)
n = len(regions)
print(n)

In [ ]:
# Economic (GDP-weighted) centroids -- the delivered reference points of the gravity
# distance matrix. Built by 02_economic_centroids.ipynb (BEA CAGDP2 county GDP + the
# 2020 Census county population centroids). Loaded here as {abbr: (lat, lon)}.
_cen = pd.read_csv(ROOT / "data/interim/economic_centroids.csv")
COORDS = {r.abbr: (r.lat, r.lon) for r in _cen.itertuples(index=False)}


In [ ]:
# distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
# Distance matrix between all pairs of regions, great-circle (haversine) between the
# GDP-weighted ECONOMIC CENTROIDS of each region (data/interim/economic_centroids.csv,
# built in 02_economic_centroids.ipynb). These are the delivered reference points; the
# earlier capital-based prior and the alternative (geometric, population-weighted)
# centroids are compared in analysis/distance_variants.py of the source project.
D= pd.DataFrame(index=regions, columns=regions, dtype=float)
for i, j in product(regions, regions):
    D.loc[i, j] = np.nan if i == j else haversine(*COORDS[i], *COORDS[j])

D_np = D.values.copy()

missing = [r for r in regions if r not in COORDS]
if missing:
    print(f"Warning: regions without coordinates: {missing}")
print(f"{len(regions)} regions | distance range: "
      f"{D_np[~np.isnan(D_np)].min():.0f}-{D_np[~np.isnan(D_np)].max():.0f} km")


In [ ]:
# Example: load use matrix for New York 2017 as a numpy array (goods × sectors)
def load_matrix(param_name, year, regions, sectors):
    df = params[param_name]
    dim = 'g' if 'g' in df.columns else 's'
    return (df[df['yr'] == year]
            .groupby(['r', dim])['value'].sum()
            .unstack(dim)
            .reindex(index=regions, columns=sectors, fill_value=0.0)
            .fillna(0.0)          
            .values)

In [ ]:
def load_year_data(year, regions, sectors, names_params):
    """Load all IO matrices for a given year. Returns a dict of arrays."""
    n, S = len(regions), len(sectors)

    names = names_params
    mats = {name: load_matrix(name, year, regions, sectors) for name in names}

    absorption = mats['dd0_'] + mats['nd0_'] + mats['m0_']
    absorption_safe = np.where(absorption < 1e-10, 1.0, absorption)

    id0_tensor = (params['id0_'][params['id0_']['yr'] == year]
                  .groupby(['r', 'g', 's'])['value'].sum()
                  .unstack('s')
                  .reindex(pd.MultiIndex.from_product([regions, sectors], names=['r', 'g']),
                           fill_value=0.0)
                  .reindex(columns=sectors, fill_value=0.0)
                  .fillna(0.0)
                  .values
                  .reshape(n, S, S))

    return {**mats, 'absorption': absorption, 'absorption_safe': absorption_safe,
            'id0': id0_tensor}

In [ ]:
EXCLUDED = {'fen', 'sle'}
sectors = sorted(s for s in params['xn0_']['g'].unique() if s not in EXCLUDED)

In [ ]:
# Index maps and dimensions
region_to_idx = {r: i for i, r in enumerate(regions)}
sector_to_idx = {s: i for i, s in enumerate(sectors)}
n, S = len(regions), len(sectors)
print(f'{n} regions x {S} sectors')

## v2 functions

In [ ]:
def ras_robust(seed, X, M, max_iter=2000, tol=1e-8):
    """RAS with convergence tracking. RAS lets us preserve the table's initial aggregate structure at the start.
    Indeed the approximation of the formula T = X.M.D^gamma distorts the matrix and does not guarantee that the resulting table
    is consistent with the aggregate flows observed at the start,
    i.e. that the total of what leaves as good g from state i toward states j equals the exports of state i for good g
    toward the NP in the initial table (row agreement);
    and that the total of what enters as good g into state j from states i equals the imports of state j for good g
    from the NP in the initial table (column agreement).

    The RAS algorithm proportionally scales up a whole row and a whole column at each iteration,
    until the row and column totals are close enough to the target totals (X and M).

    Args:
    seed: starting matrix (nxn)
    X: vector of row totals (n,)
    M: vector of column totals (n,)
    max_iter: maximum number of iterations
    tol: convergence tolerance

    Returns:
        T: adjusted matrix
        converged: boolean indicating whether convergence was reached for each sector
        final_err: final error (max of the deviations from the totals)
        iters: number of iterations performed
        initial_err: initial error (before adjustment)
    """
    T = seed.copy().astype(float) #starting nxn matrix created from the gravity seed with the chosen gamma
    r = X.values if hasattr(X, 'values') else X
    c = M.values if hasattr(M, 'values') else M
    initial_err = max(np.abs(T.sum(axis=1) - r).max(),
                      np.abs(T.sum(axis=0) - c).max())
    for it in range(max_iter):
        rs = T.sum(axis=1); rs[rs == 0] = 1
        T *= (r / rs)[:, None]
        cs = T.sum(axis=0); cs[cs == 0] = 1
        T *= (c / cs)[None, :]
        err = max(np.abs(T.sum(axis=1) - r).max(),
                  np.abs(T.sum(axis=0) - c).max())
        if err < tol:
            return T, True, err, it + 1, initial_err
    return T, False, err, max_iter, initial_err

In [ ]:
def reconstruct_bilateral_2(xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions,
                          D_np, gamma=1.0, imbalance_skip=0.50):
    """
    Reconstruct bilateral trade matrices T(region i→ region j, good g) via gravity model + RAS.
    Import target = nd0 + nm0 (direct absorption + margin absorption).
    Export target = xn0 (exports to national pool).
    Returns :
    T_all (dict sector→n×n array, n=regions) 
    df_log (convergence log).
    """
    n = len(regions)

    with np.errstate(divide='ignore', invalid='ignore'):
        friction = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma))

    T_all = {}
    log_ras = []

    for g in sectors:
        g_i = sector_to_idx[g]
        X_g = pd.Series(xn0_mat[:, g_i] , index=regions)
        M_g = pd.Series(nd0_mat[:, g_i] + nm0_mat[:, g_i], index=regions) # for margin sectors, a part of their production is absorbed as margin (nm0) rather than direct use (nd0), but both contribute to the "import" side that RAS should match
        total_X = X_g.sum()
        total_M = M_g.sum()

        imbalance = abs(total_X - total_M) / total_X
        if total_M < 1e-10 or imbalance > imbalance_skip:
            T_all[g] = np.zeros((n, n))
            log_ras.append({'sector': g, 'status': 'skipped_imbalance',
                            'err': imbalance, 'iters': 0})
            continue

        #if imbalance > 1e-6:
            #M_g = M_g * (total_X / total_M)

        seed = np.outer(X_g.values, M_g.values) * friction
        seed += 1e-8 * np.outer(X_g.values / total_X, M_g.values / M_g.sum())
        np.fill_diagonal(seed, 0.0)

        T_g, converged, err, iters, initial_err = ras_robust(seed, X_g, M_g)
        
        tol_soft = 1e-6  # relaxed tolerance for borderline cases
        if converged:
            status = 'ok'
        elif err < tol_soft:
            status = 'ok_soft'   # converged to relaxed tolerance
        else:
            status = 'FAILED'

        T_all[g] = T_g
        log_ras.append({'sector': g, 'status': status, 'initial_err': initial_err,
                        'err': err, 'iters': iters, 'seed': 'gravity'})
        

    return T_all, pd.DataFrame(log_ras)


In [ ]:
def compute_use_shares_2(id0_df, cd0_mat, i0_mat, g0_mat, a0_mat=None):
    """
    Compute use shares (intermediate + final demand) at PURCHASER prices.

    For each (region r, good g):
        use_share_interm[r, g, s] = id0[r, g, s] / total_demand[r, g]
        use_share_C/I/G[r, g]     = cd0/i0/g0[r, g] / total_demand[r, g]
    with total_demand = id0.sum_s + cd0 + i0 + g0  (purchaser-price absorption).
    By construction the shares sum to 1 over {sectors s} + {C, I, G}.


    Fallback for margin goods where all demand is zero (id0=cd0=i0=g0=0):
    use_share_interm is set proportional to each sector's total intermediate
    purchases so that nm0/dm0 flows are not lost in build_Z.

    Arguments
    ---------
    id0_df  : (n, S, S) intermediary demand by (r,g,s)
    cd0_mat : (n, S)    household consumption demand by (r,g)
    i0_mat  : (n, S)    investment demand by (r,g)
    g0_mat  : (n, S)    government consumption demand by (r,g)
    a0_mat  : ignored (deprecated -- see note above)
    """
    # Purchaser-price absorption of good g in region r:
    # total intermediate demand (sum over buying sectors s) + final demand.
    total_demand = id0_df.sum(axis=2) + cd0_mat + i0_mat + g0_mat  # (n, S)
    safe = np.where(total_demand < 1e-10, 1.0, total_demand)       # (n, S)

    use_share_interm = id0_df  / safe[:, :, None]   # (n, S, S)
    use_share_C      = cd0_mat / safe               # (n, S)
    use_share_I      = i0_mat  / safe               # (n, S)
    use_share_G      = g0_mat  / safe               # (n, S)

    # --------------------------------------------------------------------------
    # FALLBACK FOR MARGIN GOODS WITH NO RECORDED DEMAND (id0=cd0=i0=g0=0).
    #
    # Margin goods (trade/transport) carry no demand of their own, so
    # total_demand(r,g)=0 and every use share above collapses to 0. The shares
    # would then sum to 0 and the margin flows (nm0/dm0), which build_Z spreads
    # through use_share_interm, would vanish from the table.
    #
    # Lacking any signal on who buys the margin good, we route it ENTIRELY to
    # intermediate use (0% to final demand) and split it across absorbing
    # sectors using the region's average intermediate-purchase profile.
    # --------------------------------------------------------------------------

    # Per sector s: its total intermediate demand across all goods g
    # (= column sum of the intermediate matrix). Shape (n, S).
    total_inputs = id0_df.sum(axis=1)

    # Region-wide total of intermediate purchases. Shape (n, 1).
    row_sum = total_inputs.sum(axis=1, keepdims=True)

    # "Representative buyer" profile: each sector's share of the region's total
    # intermediate purchases. Sums to 1 over s. Shape (n, S).
    fallback = total_inputs / np.where(row_sum < 1e-10, 1.0, row_sum)

    # Goods with zero total demand = the margin goods to patch. Shape (n, S, 1).
    mask_zero = (total_demand < 1e-10)[:, :, None]

    # For those goods only, overwrite the (over-s) interm shares with `fallback`
    # broadcast across the g axis. The interm shares now sum to 1 -> 100% of the
    # flow goes to intermediate demand; C/I/G stay 0 -> 0% to final demand.
    use_share_interm = np.where(mask_zero, fallback[:, np.newaxis, :], use_share_interm)

    return use_share_interm, use_share_C, use_share_I, use_share_G


In [ ]:
def build_Z(dd0_mat, nd0_mat, use_share_interm, T_all, sectors, n, S, xd0_mat=None):
    """
    xd0_mat : (n, S) supply to local market = dd0 + dm0.
              If None, falls back to dd0_mat (old behaviour, dm0 missing).
    Row sum  : xd0[r,g] + xn0[r,g] + x0[r,g]  = s0 (true WiNDC supply)
    """
    local_source = xd0_mat if xd0_mat is not None else dd0_mat

    Z_4d = np.zeros((n, S, n, S))

    for r_i in range(n):
        # xd0 = dd0 + dm0 : local margin revenue now included in row sums
        Z_4d[r_i, :, r_i, :] = local_source[r_i, :, None] * use_share_interm[r_i, :, :]

    for g_i, g in enumerate(sectors):
        T_g = T_all[g]
        ush = use_share_interm[:, g_i, :]
        Z_4d[:, g_i, :, :] += T_g[:, :, None] * ush[None, :, :]

    return Z_4d.reshape(n * S, n * S)


In [ ]:
def build_F(dd0_mat, T_all, use_share_C, use_share_I, use_share_G, sectors, n, S, xd0_mat=None):
    """
    Build final demand matrix F (n·S × n·3).
    Rows: (origin region, good). Columns: (destination region, {C, I, G}).
    """
    local_src = xd0_mat if xd0_mat is not None else dd0_mat  # ← only change
    
    F_4d = np.zeros((n, S, n, 3))

    for r_i in range(n):
        F_4d[r_i, :, r_i, 0] = local_src[r_i, :] * use_share_C[r_i, :]
        F_4d[r_i, :, r_i, 1] = local_src[r_i, :] * use_share_I[r_i, :]
        F_4d[r_i, :, r_i, 2] = local_src[r_i, :] * use_share_G[r_i, :]

    for g_i, g in enumerate(sectors):
        T_g = T_all[g]
        F_4d[:, g_i, :, 0] += T_g * use_share_C[:, g_i][None, :]  # fix: += not =
        F_4d[:, g_i, :, 1] += T_g * use_share_I[:, g_i][None, :]
        F_4d[:, g_i, :, 2] += T_g * use_share_G[:, g_i][None, :]

    return F_4d.reshape(n * S, n * 3)


def build_VA_EX(ld0_mat, kd0_mat, x0_mat, n, S):
    """Value added (labor + capital) and international exports, both flattened to n·S."""
    VA = (ld0_mat + kd0_mat).reshape(n * S)
    EX = x0_mat.reshape(n * S)
    return VA, EX


## Construction

In [ ]:
YEAR = '2017'
names = ['dd0_', 'nd0_', 'xn0_', 'xd0_', 'x0_', 'm0_',
         'cd0_', 'i0_', 'g0_', 'ld0_', 'kd0_', 'ty0_']
data = load_year_data(YEAR, regions, sectors, names)
dd0_mat = data['dd0_']; nd0_mat = data['nd0_']
xn0_mat = data['xn0_']; xd0_mat = data['xd0_']; x0_mat = data['x0_']
m0_mat  = data['m0_'];  cd0_mat = data['cd0_']
i0_mat  = data['i0_'];  g0_mat  = data['g0_']
ld0_mat = data['ld0_']; kd0_mat = data['kd0_']
id0_df  = data['id0']
# national margin supply aligned (r, g), summed over margin types
nm0_mat = (params['nm0_'][params['nm0_']['yr'] == YEAR]
           .groupby(['r', 'g'])['value'].sum().unstack('g')
           .reindex(index=regions, columns=sectors, fill_value=0.0).fillna(0.0).values)
print('data loaded for', YEAR)

In [ ]:
# Bilateral reconstruction (v2): margin demand nm0 is AGGREGATED with import
# demand nd0 into a single column target, so the reconstructed flow T_new carries
# goods and margins together (one spatial distribution, gravity + RAS).
T_new, log_new = reconstruct_bilateral_2(
    xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions, D_np,
    gamma=1.0, imbalance_skip=0.50)
print('reconstructed sectors:', sum(T_new[g].sum() > 0 for g in sectors), '/', S)

In [ ]:
# Aggregated construction: margins folded into the goods flows.
#   inter-state : single combined bilateral flow T_new (column = nd0 + nm0)
#   local       : xd0 = dd0 + dm0 as the local source (folds in domestic margins)
#   use shares  : a SINGLE set -- goods and margins share the same allocation key
us_int, us_C, us_I, us_G = compute_use_shares_2(id0_df, cd0_mat, i0_mat, g0_mat)

Z = build_Z(dd0_mat, nd0_mat, us_int, T_new, sectors, n, S, xd0_mat=xd0_mat)
F = build_F(dd0_mat, T_new, us_C, us_I, us_G, sectors, n, S, xd0_mat=xd0_mat)
VA = (ld0_mat + kd0_mat).reshape(n * S)
EX = x0_mat.reshape(n * S)
print(f'Z {Z.shape} sum={Z.sum():.1f} | F {F.shape} sum={F.sum():.1f} | VA sum={VA.sum():.1f} | EX sum={EX.sum():.1f}')

## Diagnostic

In [ ]:
# -- DIAGNOSTIC: reference-equilibrium identities --------------------------------
Y = (params['ys0_'][params['ys0_']['yr'] == YEAR].groupby(['r', 's'])['value'].sum()
     .unstack('s').reindex(index=regions, columns=sectors, fill_value=0.0)
     .fillna(0.0).values.reshape(n * S))
ty0_mat  = load_matrix('ty0_', YEAR, regions, sectors)
tax_flat = (ty0_mat * Y.reshape(n, S)).reshape(n * S)
M_imp    = (m0_mat[:, :, None] * us_int).sum(axis=1).reshape(n * S)   # ROW imports, single use shares
s0       = (xd0_mat + xn0_mat + x0_mat)

# Row (supply) identity:  row(Z) + row(F) + EX = xd0 + xn0 + x0
row_sum = (Z.sum(axis=1) + F.sum(axis=1) + EX).reshape(n, S)
ras_err_mat = s0 - row_sum
# Column (cost) identity:  Y = colZ + imports + VA + taxes + residual
col_Z    = Z.sum(axis=0)
residual = Y - (col_Z + M_imp + VA + tax_flat)
mask = Y > 0.1

print('=== Row identity:  row(Z)+row(F)+EX = xd0+xn0+x0 ===')
print(f'  S row_sum {row_sum.sum():.1f}  vs  S s0 {s0.sum():.1f}  | gap {ras_err_mat.sum():+.2f} '
      f'({ras_err_mat.sum()/s0.sum()*100:+.3f}%)  | S|err|/Ss0 {np.abs(ras_err_mat).sum()/s0.sum()*100:.3f}%')
print('=== Column identity:  Y = colZ + imports + VA + taxes + residual ===')
print(f'  S check {(col_Z+M_imp+VA+tax_flat).sum():.1f}  vs  S Y {Y.sum():.1f}  | resid {residual.sum():+.1f} '
      f'({residual.sum()/Y.sum()*100:+.2f}%)  | S|err|/SY {np.abs(residual[mask]).sum()/Y[mask].sum()*100:.2f}%')

## Analyse

In [ ]:
# -- ANALYSE: residual by sector -------------------------------------------------
import matplotlib.pyplot as plt

def _by_sector(vec_nS, denom_nS, thr=0.1):
    out = np.zeros(S)
    Mv = vec_nS.reshape(n, S); Dn = denom_nS.reshape(n, S)
    for s in range(S):
        m = Dn[:, s] > thr
        out[s] = np.abs(Mv[m, s]).sum() / Dn[m, s].sum() * 100 if m.any() else 0.0
    return out

row_err_s = _by_sector(ras_err_mat, s0)
col_err_s = _by_sector(residual, Y)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))
for ax, err, title in [(axes[0], row_err_s, 'Row identity residual by sector (% of s0)'),
                       (axes[1], col_err_s, 'Column identity residual by sector (% of Y)')]:
    order = np.argsort(err)[::-1]
    ax.bar(range(S), err[order], color=['#d62728' if e > 5 else '#1f77b4' for e in err[order]])
    ax.axhline(5, color='orange', ls='--', lw=1)
    ax.set_xticks(range(S)); ax.set_xticklabels([sectors[i] for i in order], rotation=90, fontsize=7)
    ax.set_ylabel('%'); ax.set_title(title)
plt.tight_layout(); plt.show()

print('Top row-residual sectors:', [sectors[i] for i in np.argsort(row_err_s)[::-1][:5]])
print('Top col-residual sectors:', [sectors[i] for i in np.argsort(col_err_s)[::-1][:5]])